# Guided tour of the financials factor pipeline

An end-to-end walk from **raw data → panel → factors → validation & selection**, using
the real modules under `src/`. The emphasis is the final stage: turning ~20 raw style
factors into a **shortlisted, non-redundant set** via Information Coefficient, Fama-MacBeth,
redundancy clustering, and a lasso.

The strategy universe is **financials (banks + insurers)**. Banks and insurers carry
mutually-exclusive metrics, so the pipeline is *sub-universe aware* throughout: factors are
partitioned by `Applicability`, and the cross-sectional steps never mix the two sleeves.

> **Scope knobs.** The full panel is ~150M price rows. To keep the tour to well under a
> minute we restrict to a multi-region set of countries and a 10-year window (set below).
> Widen `COUNTRIES` / `WINDOW` to run larger slices — the code is identical.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, functools, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl
from plotnine import (ggplot, aes, geom_col, geom_line, geom_point, geom_hline,
                      coord_flip, facet_wrap, scale_fill_manual, scale_x_continuous,
                      labs, theme_bw, theme, element_text)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config, universe
from src.data import loaders, joins, preprocess
from src.factors.base import registry
from src.validation import single_factor as sf, redundancy as rd, neutralization as nz

cfg = config.load(PROJECT_ROOT / "config.yaml")
cfg.raw["data"]["root"] = str(PROJECT_ROOT / cfg["data"]["root"])   # anchor relative paths

# ── tour knobs ────────────────────────────────────────────────────────────────
COUNTRIES = ["US", "JP", "GB", "FR", "DE", "HK", "AU", "KR"]
WINDOW = ("2000-01-01", "2019-12-31")
W = (pl.lit(WINDOW[0]).str.to_date(), pl.lit(WINDOW[1]).str.to_date())
pl.Config.set_tbl_rows(30)

## 1. Ingestion

`loaders.load_all` returns every raw table as a **lazy** `scan_ipc` frame — nothing is read
until a `.collect()`. That lets us prune to the universe/window before materialising anything.

In [ ]:
raw = loaders.load_all(cfg)
print("tables:", sorted(raw))
raw["price"].collect_schema()   # daily price / return / mcap

## 2. Point-in-time panels & universe

We build **two** panels, both lazy:

* `joins.build_market_frame` — a *light, full-universe* frame (all ~40k stocks, only the
  columns the cap-weighted **factor-mimicking portfolios** need) plus a `tradeable` flag and
  the `industry` label. Market and Country returns span every stock; only the ~2k tradeable
  names ever get an *exposure* attached.
* `joins.build_sector_panel` — a *tradeable-only, rich* panel. It filters `price` to the
  tradeable `stock_id` set (`universe.tradable_ids`) **before** the fundamentals as-of join,
  so that join and all style-factor work run on ~2k stocks, not 40k.

`preprocess.clean` (return imputation, drop rows with no risk-free rate, excess return) is
applied to each. We pre-filter the heavy `price` / `zero_curve` tables to the tour's ids,
window and currencies first.

In [ ]:
sm_lazy = raw["security_master"].filter(pl.col("country_code").is_in(COUNTRIES))

raw["price"] = (
    raw["price"]
    .filter(pl.col("date").is_between(*W))
    .join(sm_lazy.select("stock_id"), on="stock_id", how="inner")
)

ccys_lazy = sm_lazy.select("currency_code").unique()
raw["zero_curve"] = (
    raw["zero_curve"]
    .filter(pl.col("date").is_between(*W))
    .join(ccys_lazy, left_on="currency", right_on="currency_code", how="inner")
)

# Full-universe light frame (mimicking portfolios) + tradeable-only rich panel (style factors).
market_frame = preprocess.clean(joins.build_market_frame(raw, cfg), cfg)
sector_panel = preprocess.clean(joins.build_sector_panel(raw, cfg), cfg)

n_all  = market_frame.select(pl.col("stock_id").n_unique()).collect().item()
n_trad = market_frame.filter(pl.col("tradeable")).select(pl.col("stock_id").n_unique()).collect().item()
print(f"market frame: {n_all:,} stocks in slice  |  tradeable universe: {n_trad:,}")
sector_panel.group_by("industry").len().sort("len", descending=True).collect()

## 3. Factor generation

The registry enumerates every factor; each `compute` returns a uniform
`[stock_id, date, <shorthand>]` frame. **Style** factors are cross-sectional ratios (computed
on the daily tradeable panel, since momentum/volatility need the daily history); **non-style**
structural factors (Market / Country / Industry) are per-security *loadings* — rolling
time-series betas on their factor-mimicking-portfolio returns, estimated on **monthly**
observations to match the quarterly rebalance.

Because the portfolio only rebalances quarterly, we then **downsample both to the rebalance
grid** (`preprocess.rebalance_grid` / `to_rebalance`): each stock's period-end row, tagged with
a common `period` key so the cross-sectional steps below never fragment under staggered
trading calendars.

In [ ]:
reg = registry()
STYLE    = {s: c for s, c in reg.items() if c.__module__.startswith("src.factors.style")}
NONSTYLE = {s: c for s, c in reg.items() if c.__module__.startswith("src.factors.nonstyle")}
SLEEVE   = {s: c.sleeve for s, c in reg.items()}
print("style   :", sorted(STYLE))
print("nonstyle:", sorted(NONSTYLE))

def join_all(frames):
    return functools.reduce(lambda a, b: a.join(b, on=["stock_id", "date"], how="left"), frames)

grid = preprocess.rebalance_grid(sector_panel, cfg)
daily_grid = sector_panel.select("stock_id", "date")

# 1. Non-style loadings: monthly betas (mimicking return spans the full universe,
#    exposure only on the tradeable names), downsampled to the rebalance grid.
allload = preprocess.to_rebalance(
    join_all([daily_grid] + [NONSTYLE[s]().compute(market_frame, cfg) for s in NONSTYLE]),
    grid,
).collect()

# 2. Style scores: computed daily (momentum/vol need the daily history), then
#    downsampled to the same grid. `period` keys the cross-sections below.
scores = preprocess.to_rebalance(
    join_all([sector_panel.select("stock_id", "date", "industry")]
             + [STYLE[s]().compute(sector_panel, cfg) for s in STYLE]),
    grid,
).collect()

print("style scores:", scores.shape, "| non-style loadings:", allload.shape)

The neutralization regressors are the **non-style structural loadings**: the market beta
`MKT`, plus — for each country/industry group — that group's own slope `beta_{group}` (0
off-group) and a one-hot baseline dummy `is_{group}`. They must be well-populated
to serve as regressors (a pooled cross-sectional OLS drops any row with a null regressor), so
we confirm coverage first, then neutralize every style score against the whole design.

In [ ]:
lcols = [c for c in allload.columns if c not in ("stock_id", "date", "period")]
coverage = (pl.DataFrame({"loading": lcols})
            .with_columns(pct_populated=pl.Series(
                [round(allload[c].drop_nulls().len() / allload.height, 3) for c in lcols]))
            .sort("pct_populated", descending=True))
loadings = allload

coverage

In [ ]:
from src.data.preprocess import winsorize_cross_section
import polars as pl

continuous_loadings = [c for c in loadings.columns if c == "MKT" or c.startswith("beta_")]
beta_cols = [c for c in continuous_loadings if c.startswith("beta_")]

# 1. Mask structural zeros (exactly 0.0) to Null so Polars ignores them in quantiles
mask_exprs = [
    pl.when(pl.col(c) != 0.0).then(pl.col(c)).otherwise(None).alias(c)
    for c in beta_cols
]
loadings_masked = loadings.with_columns(mask_exprs)

# 2. Apply winsorization cross-sectionally per rebalance period
limits = cfg["preprocess"]["winsorize_limits"]
loadings_winsorized = winsorize_cross_section(
    loadings_masked,
    cols=continuous_loadings,
    limits=limits,
    by="period"
)

# 3. Restore structural zeros (fill Nulls back to 0.0) for downstream neutralizations
restore_exprs = [
    pl.col(c).fill_null(0.0).alias(c)
    for c in beta_cols
]
loadings = loadings_winsorized.with_columns(restore_exprs)

In [ ]:
from plotnine import geom_histogram, ggplot, aes, facet_wrap, labs, theme_bw, theme, element_text
import polars as pl

# Identify continuous structural loading columns
id_cols = ["stock_id", "date", "period"]
loading_cols = [c for c in loadings.columns if c not in id_cols and (c == "MKT" or c.startswith("beta_"))]
beta_cols = [c for c in loading_cols if c.startswith("beta_")]

# Mask structural zeros (which are exactly 0.0) to Null
mask_exprs = [
    pl.when(pl.col(c) != 0.0).then(pl.col(c)).otherwise(None).alias(c)
    for c in beta_cols
]
loadings_plot_df = loadings.with_columns(mask_exprs)

# Melt (unpivot) the polars DataFrame and convert to pandas for plotnine
loadings_melted = (
    loadings_plot_df
    .unpivot(index=[c for c in id_cols if c in loadings_plot_df.columns], 
             on=loading_cols, 
             variable_name="loading", 
             value_name="value")
    .drop_nulls("value")  # Successfully drops all structural zeros
    .to_pandas()
)

# Plot the marginal distributions in a faceted grid
loadings_plot = (
    ggplot(loadings_melted, aes(x="value"))
    + geom_histogram(bins=40, fill="#d95f02", alpha=0.8)
    + facet_wrap("~loading", ncol=4, scales="free")
    + labs(title="Marginal Distributions of Structural Factor Loadings (Excl. Structural Zeros)",
           x="Estimated Beta", 
           y="Count")
    + theme_bw()
    + theme(figure_size=(14, 12), 
            subplots_adjust={"hspace": 0.55, "wspace": 0.3},
            strip_text=element_text(size=8), 
            axis_text=element_text(size=6))
)

loadings_plot

## 4. Regularize & neutralize

`preprocess.regularize` winsorizes → median-imputes → z-scores **within each factor's
sub-universe** (a bank never receives an insurer's median, and jointly-computable factors like
`B/P` are nulled outside their sub-universe). `neutralization.neutralize` then, per **period**
cross-section:

1. **imputes** any missing structural loading (a rolling-beta warms up over ~2y) with the
   cross-sectional median of its comparable group — otherwise `null_policy="drop"` would delete
   a newly listed stock's *valid* style scores along with its missing betas (the "two-year
   blackout");
2. **regresses** each style score on the structural design (market + per-group country/industry
   betas and their baseline dummies) and keeps the residual — the part *not* explained by
   market/country/industry risk;
3. **re-standardizes** the residual back to unit variance (an OLS residual has variance
   `1 − R²`, not 1), so neutralized scores stay comparable and correctly weighted.

Both `regularize` and `neutralize` group on the `period` key (one cross-section per rebalance),
so staggered period-end dates don't split a rebalance across cross-sections.

In [ ]:
regd = preprocess.regularize(scores, cfg)               # cross-sections per `period`
neu  = nz.neutralize(regd, loadings, cfg, by="period")       # keeps `period` for redundancy below

# sub-universe partitioning in action: B/P is an insurance factor (computable for banks too),
# so it is present for insurers and null for banks after regularisation.
bp_bank = neu.filter(pl.col("industry") == "bank")["B/P"].drop_nulls().len()
bp_ins  = neu.filter(pl.col("industry").str.starts_with("insurance"))["B/P"].drop_nulls().len()
print(f"neutralized {neu.shape}; B/P non-null  banks={bp_bank}  insurers={bp_ins}")
print("(loadings warm up over ~2y, so early residuals are null and drop out of validation)")

In [ ]:
from plotnine import geom_histogram
# Identify the style factor columns (all columns except the identifiers)
id_cols = ["stock_id", "date", "period", "industry"]
factor_cols = [c for c in neu.columns if c not in id_cols]

# Melt (unpivot) the polars DataFrame and convert to pandas for plotnine
neu_melted = (
    neu
    .unpivot(index=[c for c in id_cols if c in neu.columns], 
             on=factor_cols, 
             variable_name="factor", 
             value_name="score")
    .drop_nulls("score")
    .to_pandas()
)

# Plot the marginal distributions in a faceted grid
marginal_plot = (
    ggplot(neu_melted, aes(x="score"))
    + geom_histogram(bins=40, fill="#2c7fb8", alpha=0.8)
    + facet_wrap("~factor", ncol=4, scales="free_y")
    + labs(title="Marginal Distributions of Neutralized Style Scores",
           x="Neutralized Score (Z-Score)", 
           y="Count")
    + theme_bw()
    + theme(figure_size=(14, 16), 
            subplots_adjust={"hspace": 0.55, "wspace": 0.3},
            strip_text=element_text(size=8), 
            axis_text=element_text(size=6))
)

marginal_plot

## 5. Validation & selection

This is the heart of the tour. Five complementary lenses, then a combined scorecard.

### 5a. Information Coefficient & IC decay
Rank IC = per-period Spearman correlation between the neutralized exposure and the forward
quarterly return. The forward return is itself **residualised against the structural loadings**
(`nonstyle_exposures=loadings`) so we correlate a market-orthogonal score against a
market-orthogonal target — otherwise the raw return's un-modelled market risk (the very risk
the scores were neutralized against) would dominate the ranks. Its mean/std is the **IC-IR**; a
factor is shortlisted at `|IR| > 0.3`. Horizons `1,2,3` (quarters) trace the **decay** curve.

In [ ]:
WINS = cfg["preprocess"]["winsorize_limits"]   # clip daily-return outliers before compounding
ic = sf.rank_ic(neu, sector_panel.select("stock_id", "date", "excess_return"),
                lags=tuple(cfg["validation"]["ic"]["decay_lags"]),
                nonstyle_exposures=loadings, winsorize_limits=WINS)   # fwd returns net of non-style risk
ir = sf.information_ratio(ic)
THR = cfg["validation"]["ic"]["ir_shortlist_threshold"]

ir1 = ir.filter(pl.col("lag") == 1).drop_nulls("ir").sort(pl.col("ir").abs(), descending=True)
shortlist_ir = set(ir1.filter(pl.col("ir").abs() > THR)["factor"])
print(f"IR shortlist (|IR| > {THR}):", sorted(shortlist_ir))
ir1.head(10)

In [ ]:
ir1p = ir1.with_columns(shortlisted=pl.col("ir").abs() > THR).to_pandas()
(ggplot(ir1p, aes("reorder(factor, ir)", "ir", fill="shortlisted"))
 + geom_col() + coord_flip()
 + geom_hline(yintercept=[-THR, THR], linetype="dashed")
 + labs(title="IC-IR by factor (lag 1 = one quarter ahead)", x="", y="IC-IR")
 + theme_bw())

In [ ]:
decay = (ic.filter(pl.col("factor").is_in(sorted(shortlist_ir)))
         .group_by("factor", "lag").agg(ic=pl.col("ic").mean()).sort("factor", "lag").to_pandas())
(ggplot(decay, aes("lag", "ic", color="factor"))
 + geom_line() + geom_point()
 + labs(title="IC decay of shortlisted factors", x="horizon (quarters)", y="mean IC")
 + theme_bw())

### 5b. Quantile return profiling
Rank IC is a *linear* rank measure, so a factor can post a strong IC yet respond **non-linearly** — only the tails pay, or the middle humps. Here `quantile_returns` buckets each factor's neutralized exposure into deciles cross-sectionally per rebalance period (`validation.quantiles`), forms the equal-weighted **forward excess return** of each decile portfolio, and averages over periods (same pooled cross-section as the IC, so a sector factor is bucketed only within its own sub-universe). A clean linear factor traces a monotone ramp from decile 1 (lowest exposure) to decile *N* (highest); a kink, hump, or tails-only spike flags non-linearity that the IC would miss. Panels are ordered by |IC-IR| and the **IC-IR shortlisted** factors are highlighted; the y-axis is free per panel so each shape is legible.

In [ ]:
QN = cfg["validation"]["quantiles"]
qr = sf.quantile_returns(neu, sector_panel.select("stock_id", "date", "excess_return", "mcap_usd"),
                         nonstyle_exposures=loadings, winsorize_limits=WINS,
                         n_quantiles=QN, lag=1)
print(f"{QN}-quantile profiles for {qr['factor'].n_unique()} factors "
      f"(up to {qr['n_periods'].max()} rebalance periods each)")

# order panels by |IC-IR| so the significant factors sit top-left; highlight them
order = ir1.sort(pl.col("ir").abs(), descending=True)["factor"].to_list()
order += [f for f in qr["factor"].unique() if f not in order]   # any factor without an IR
qrp = (qr
       .with_columns(
           status=pl.when(pl.col("factor").is_in(list(shortlist_ir)))
                    .then(pl.lit("IC-IR shortlisted")).otherwise(pl.lit("not shortlisted")),
           ret_pct=pl.col("mean_ret") * 100,
           factor=pl.col("factor").cast(pl.Enum(order)))
       .to_pandas())

(ggplot(qrp, aes("quantile", "ret_pct", fill="status"))
 + geom_col()
 + geom_hline(yintercept=0, size=0.3)
 + facet_wrap("~factor", ncol=4, scales="free_y")
 + scale_fill_manual(values={"IC-IR shortlisted": "#2c7fb8", "not shortlisted": "#c9c9c9"})
 + scale_x_continuous(breaks=[1, QN])
 + labs(title=f"Decile forward-return profiles ({QN} buckets, one quarter ahead)",
        subtitle="bucket 1 = lowest exposure … bucket N = highest exposure",
        x="exposure decile", y="mean quarterly excess return (%)", fill="")
 + theme_bw()
 + theme(figure_size=(14, 11), subplots_adjust={"hspace": 0.55, "wspace": 0.3},
         strip_text=element_text(size=8), axis_text=element_text(size=6),
         legend_position="top"))

### 5c. Fama-MacBeth premia
Per-period cross-sectional regressions of forward returns on the factors, **run separately per
sub-universe** (banks and insurers hold disjoint factors), aggregated with Newey-West t-stats.

In [ ]:
fm = sf.fama_macbeth(neu, sector_panel.select("stock_id", "date", "excess_return"),
                     winsorize_limits=WINS).filter(pl.col("factor") != "const")
fm_sig = set(fm.filter(pl.col("t_stat").abs() > 2)["factor"])
print("FM significant (|t| > 2):", sorted(fm_sig))
fm.sort(pl.col("t_stat").abs(), descending=True, nulls_last=True).head(30)

In [ ]:
fmp = fm.drop_nulls("t_stat").to_pandas()
(ggplot(fmp, aes("reorder(factor, t_stat)", "t_stat"))
 + geom_col() + coord_flip() + facet_wrap("sub_universe", scales="free_y")
 + geom_hline(yintercept=[-2, 2], linetype="dashed")
 + labs(title="Fama-MacBeth premia (Newey-West t)", x="", y="t-stat")
 + theme_bw())

In [ ]:
from plotnine import geom_errorbar

# Extract to pandas and calculate the 2 stdv confidence bounds
fmp_ci = fm.drop_nulls("mean_coef").to_pandas()
fmp_ci["ci_lower"] = fmp_ci["mean_coef"] - 1 * fmp_ci["nw_se"]
fmp_ci["ci_upper"] = fmp_ci["mean_coef"] + 1 * fmp_ci["nw_se"]

# Plot point estimates and confidence windows, ordered by highest point estimate
(ggplot(fmp_ci, aes(x="reorder(factor, mean_coef)", y="mean_coef", color="sub_universe"))
 + geom_hline(yintercept=0, linetype="dashed", color="black", alpha=0.6)
 + geom_errorbar(aes(ymin="ci_lower", ymax="ci_upper"), width=0.2)
 + geom_point(size=2)
 + coord_flip()
 + labs(title="Fama-MacBeth Factor Premia & Error Bars", 
        x="", 
        y="Mean Coefficient Estimate",
        color="Sub-universe")
 + theme_bw()
 + theme(figure_size=(8, 8)))

### 5d. Redundancy — correlation clustering
Flag factor pairs whose time-averaged cross-sectional correlation exceeds the threshold, group
them into clusters (connected components), and keep the **highest-IR representative** of each
cluster (dropping the rest as redundant).

In [ ]:
pairs, flagged = rd.average_correlation(neu, threshold=cfg["validation"]["redundancy"]["correlation_threshold"])
clusters = rd.cluster_factors((pairs, flagged))
reps = set(rd.select_cluster_representatives(clusters, ic_ir=ir))
print("flagged pairs:", flagged.height)
print("correlated clusters:", [c for c in clusters if len(c) > 1])
print("kept representatives:", len(reps), "of", len(STYLE), "factors")
flagged

### 5e. Parsimony — lasso
A cross-validated lasso predictive regression (per sub-universe, **CV folds grouped by period**
to avoid look-ahead leakage). Like the IC, the forward-return target is first **residualised
against the structural loadings** (`nonstyle_exposures=loadings`) so the neutralized regressors
predict a market-orthogonal target, not raw returns. Survivors are the non-shrunk factors.

In [ ]:
lasso = set(rd.lasso_select(sector_panel.select("stock_id", "date", "excess_return"), neu, cfg,
                            nonstyle_exposures=loadings))   # residualise the lasso target too
print("lasso survivors:", sorted(lasso))

### 5f. Final selection scorecard
Combine the lenses: keep a factor when it is a **cluster representative** (non-redundant) *and*
has predictive evidence from **at least one** of IR / Fama-MacBeth / lasso.

In [ ]:
ir_best = ir.group_by("factor").agg(ic_ir=pl.col("ir").max()).select("factor", "ic_ir")
fm_best = (fm.sort(pl.col("t_stat").abs(), descending=True, nulls_last=True)
           .unique("factor", keep="first").select("factor", fm_t="t_stat"))

scorecard = (pl.DataFrame({"factor": sorted(STYLE)})
    .join(ir_best, on="factor", how="left")
    .join(fm_best, on="factor", how="left")
    .with_columns(
        sleeve=pl.col("factor").replace_strict(SLEEVE, default=None),
        representative=pl.col("factor").is_in(list(reps)),
        ir_pass=pl.col("factor").is_in(list(shortlist_ir)),
        fm_sig=pl.col("factor").is_in(list(fm_sig)),
        lasso=pl.col("factor").is_in(list(lasso)))
    .with_columns(keep=pl.col("representative") & (pl.col("ir_pass") | pl.col("fm_sig") | pl.col("lasso")))
    .sort("keep", pl.col("ic_ir").abs(), descending=True, nulls_last=True))

final = sorted(scorecard.filter(pl.col("keep"))["factor"])
print("FINAL SHORTLIST:", final)
scorecard

## Caveats & scaling up

* **Working slice.** Results above are for the `COUNTRIES` / `WINDOW` slice; widen them (or drop
  the pre-filter) to run the full universe — runtime scales roughly with row count.
* **Loadings warm-up.** Structural betas need ~2 years of history, so the first window years and
  the newest listings carry null loadings and drop out of neutralization/validation.
* **Yield-curve factors** are **style** factors (`Ret~*` all-financials, `NIM~*` banks,
  `FIY~*` insurance), regularized and neutralized within their sub-universe.
* **Currency factor** is not yet implemented (`src/factors/nonstyle/currency.py`).
* The IC/IR thresholds, correlation cut-off, rebalancing frequency and Newey-West lags are all
  read from `config.yaml`.